# MedBoard — Train Baseline U-Net at 512x512 (Original 90/10 Protocol)

**Run this notebook in Google Colab to train the standard Baseline U-Net on native 512x512 resolution using the exact 90/10 train-validation split from our original baseline run.**

### Experiment Objective:
- **Controlled Scientific Comparison**: Tests the exact hypothesis: *Does scaling from 256x256 to 512x512 resolution improve performance under identical experimental conditions?*
- **Data Split**: Standard **90% Train / 10% Validation** split (~3,539 train / ~394 val) with the **860 test scans kept completely separate and unseen** until final evaluation (identical to our original U-Net Run 1).
- **Model Architecture**: Standard Baseline U-Net (`base_features=32`, **7.85M parameters**).
- **Memory Optimization**: `batch_size = 2` + `gradient_accumulation_steps = 4` (effective batch size = 8). Peak VRAM stays at ~5.5 GB, well within Colab T4's 14.6 GB limit.
- **Loss Function**: `0.5 BCE + 0.5 Dice` (matching the original baseline U-Net run).
- **Reference Benchmark**: Original 256x256 Baseline U-Net achieved **86.84% Test Dice**.

**Expected training time:** ~6–8 minutes per epoch on Colab T4 GPU.

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import os, sys, shutil

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code / Jupyter)"}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone or update MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # Copy dataset from Google Drive to local Colab SSD for fast I/O
    if not os.path.exists('/content/data'):
        print('Copying dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Dataset copied.')
    else:
        print('Dataset already available on local SSD.')

    # Install requirements
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    sys.path.insert(0, '/content/MedBoard')
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'
    CONFIG_PATH = '/content/MedBoard/configs/config.yaml'

else:
    # Local paths
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.exists(os.path.join(os.getcwd(), '..', 'modules')) else os.getcwd()
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'models', 'segmentation')
    CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'config.yaml')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'unet_512_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'unet_512_brisc.pth')

print('\nSetup complete!')
print(f'  Data Path    -> {DATA_ROOT}')
print(f'  Weights Path -> {WEIGHTS_DIR}')
print(f'  Checkpoint   -> {CHECKPOINT_PATH}')

## Cell 2 — Verify GPU & Hardware Acceleration

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: No GPU detected! Please select Runtime → Change runtime type → T4 GPU')

## Cell 3 — Load BRISC 2025 Dataset (512x512 Native, 90/10 Split, Separate Test Set)

In [ ]:
from modules.seg_dataset import get_seg_dataloaders

# Standard 90/10 split matching the original baseline U-Net run
# Test set (860 scans) is kept completely separate and unseen
train_loader, val_loader, test_loader = get_seg_dataloaders(
    train_images_dir = f'{DATA_ROOT}/segmentation_task/train/images',
    train_masks_dir  = f'{DATA_ROOT}/segmentation_task/train/masks',
    test_images_dir  = f'{DATA_ROOT}/segmentation_task/test/images',
    test_masks_dir   = f'{DATA_ROOT}/segmentation_task/test/masks',
    batch_size       = 2,
    num_workers      = 2,
    val_split        = 0.1,
    seed             = 42,
    config_path      = CONFIG_PATH,
    image_size       = 512,
)

print('\nDataLoaders ready (512x512 Native Resolution, 90/10 Split):')
print(f'  Train : {len(train_loader.dataset)} samples → {len(train_loader)} batches (90% Training Data)')
print(f'  Val   : {len(val_loader.dataset)} samples → {len(val_loader)} batches (10% Internal Validation)')
print(f'  Test  : {len(test_loader.dataset)} samples → {len(test_loader)} batches (860 Separate Test Scans)')

# Verify batch shape is 512x512
sample_img, sample_mask = next(iter(train_loader))
print(f'\nSample batch shape:')
print(f'  Image: {tuple(sample_img.shape)}  (Expected: (2, 3, 512, 512))')
print(f'  Mask:  {tuple(sample_mask.shape)}  (Expected: (2, 1, 512, 512))')
assert sample_img.shape[2:] == (512, 512), 'Image resolution mismatch! Expected 512x512.'

## Cell 4 — Build Standard Baseline U-Net (7.85M Parameters)

In [ ]:
from modules.unet import UNet, count_parameters

# Standard U-Net with base_features=32 (exact architecture as baseline)
model = UNet(in_channels=3, out_channels=1, base_features=32)

p_total = count_parameters(model)
print('Architecture: Standard Baseline U-Net (Fully Convolutional)')
print(f'Total Trainable Parameters: {p_total:,} ({p_total/1e6:.2f}M)')

# Forward shape validation with 512x512 tensor
model.eval()
dummy = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = model(dummy)
print(f'\nShape Check: Input {tuple(dummy.shape)} → Output {tuple(out.shape)}')
assert out.shape == (1, 1, 512, 512), 'Output shape mismatch! Expected (1, 1, 512, 512).'
print('Shape verification passed successfully!')

## Cell 5 — Train Model at 512x512 with Gradient Accumulation

In [ ]:
from modules.seg_trainer import train_segmentation
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

print('Training Configuration:')
print('  Resolution:                   512x512 (Native BRISC)')
print('  Micro-batch size:             2')
print('  Gradient accumulation steps:  4 (Effective batch size = 8)')
print('  Optimizer:                    Adam (lr=1.0e-4, weight_decay=1e-5)')
print('  Loss function:                0.50 BCE + 0.50 Dice (Original baseline ratio)')
print('  Scheduler:                    CosineAnnealingLR (min_lr=1e-6)\n')

history = train_segmentation(
    model                       = model,
    train_loader                = train_loader,
    val_loader                  = val_loader,
    epochs                      = 35,
    patience                    = 8,
    lr_scheduler_type           = 'cosine',
    checkpoint_path             = CHECKPOINT_PATH,
    device_str                  = 'cuda',
    bce_weight                  = 0.50,
    dice_weight                 = 0.50,
    optimizer                   = optimizer,
    gradient_accumulation_steps = 4,
)

## Cell 6 — Save Final Weights to Drive

In [ ]:
import shutil

shutil.copy(CHECKPOINT_PATH, FINAL_PATH)
print(f'Final weights saved to: {FINAL_PATH}')
print(f'Best val dice: {max(history["val_dice"]):.4f}  ({max(history["val_dice"])*100:.2f}%)')

## Cell 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', color='royalblue', linewidth=2)
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
ax1.set_title('Loss per Epoch (0.50 BCE + 0.50 Dice)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice curve
ax2.plot(epochs_ran, history['train_dice'], label='Train Dice', color='royalblue', linewidth=2)
ax2.plot(epochs_ran, history['val_dice'],   label='Val Dice (10% Internal)', color='tomato', linewidth=2)
ax2.axhline(0.8684, color='gray',  linestyle=':',  alpha=0.8, label='256x256 Baseline Test (86.84%)')
ax2.axhline(0.8800, color='green', linestyle='--', alpha=0.8, label='88.0% Benchmark Target')
ax2.set_title('Dice Score per Epoch (512x512 High-Res)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice')
ax2.set_ylim([0, 1])
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('Baseline U-Net (512x512 Native Resolution) — MedBoard', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = f'{WEIGHTS_DIR}/unet_512_training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved to {plot_path}')

## Cell 8 — Evaluate on Separate Test Set (860 Scans, 512x512)

In [ ]:
from modules.seg_trainer import validate_one_epoch, DiceBCELoss
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'model' not in locals():
    model = UNet(in_channels=3, out_channels=1, base_features=32)
    model.load_state_dict(torch.load(FINAL_PATH, map_location=device))
    print(f'Model loaded from: {FINAL_PATH}')

model = model.to(device)
model.eval()

criterion = DiceBCELoss(bce_weight=0.50, dice_weight=0.50)
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)

print('=' * 65)
print('     Baseline U-Net (512x512) — Held-Out Test Set Evaluation')
print('=' * 65)
print(f'  Test Loss (0.50 BCE + 0.50 Dice): {test_loss:.4f}')
print(f'  Test Dice Score:                 {test_dice:.4f} ({test_dice*100:.2f}%)')
print('=' * 65)
print(f'  Original 256x256 Baseline:        86.84%')
print(f'  Resolution Net Gain:              {test_dice*100 - 86.84:+.2f}%')
print('=' * 65)

## Cell 9 — Optimal Threshold Sweep & Slice Breakdown

In [ ]:
thresholds = [0.30, 0.35, 0.38, 0.40, 0.42, 0.45, 0.48, 0.50, 0.55, 0.60]
dice_records = {t: [] for t in thresholds}
iou_records  = {t: [] for t in thresholds}
tumor_sizes  = []

print(f'Evaluating {len(test_loader.dataset)} separate test samples across {len(thresholds)} thresholds (512x512)...')

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        gt = masks.numpy()

        for i in range(images.size(0)):
            p = probs[i, 0]
            g = gt[i, 0]
            gt_sum = g.sum()
            tumor_sizes.append(gt_sum)

            for t in thresholds:
                pred_bin = (p >= t).astype(np.float32)
                inter = (pred_bin * g).sum()
                union = pred_bin.sum() + gt_sum

                dice = (2.0 * inter + 1.0) / (union + 1.0)
                iou  = (inter + 1.0) / (union - inter + 1.0)

                dice_records[t].append(dice)
                iou_records[t].append(iou)

print('\n' + '=' * 65)
print(f"{'Threshold':^12} | {'Mean Dice':^14} | {'Median Dice':^14} | {'Mean IoU':^12}")
print('=' * 65)

best_t = 0.50
best_mean = -1.0

for t in thresholds:
    mean_d = np.mean(dice_records[t]) * 100
    med_d  = np.median(dice_records[t]) * 100
    mean_i = np.mean(iou_records[t]) * 100

    if mean_d > best_mean:
        best_mean = mean_d
        best_t = t
        star = ' <-- BEST'
    else:
        star = ''
    print(f"    {t:0.2f}     |     {mean_d:0.2f}    % |     {med_d:0.2f}    % |     {mean_i:0.2f}    %{star}")

print('=' * 65)
print(f'Optimal Threshold: {best_t:0.2f}')
print(f'Peak Test Mean Dice: {best_mean:0.2f}%')
print(f'Median Dice at Peak: {np.median(dice_records[best_t])*100:0.2f}%')
print('=' * 65)

# Slice breakdown (at 512x512, pixel counts are 4x relative to 256x256)
tumor_sizes = np.array(tumor_sizes)
small_mask = (tumor_sizes > 0) & (tumor_sizes < 2000)       # <500 px equivalent at 256x256
medium_mask = (tumor_sizes >= 2000) & (tumor_sizes < 10000) # 500-2500 px equivalent
large_mask = tumor_sizes >= 10000                           # >2500 px equivalent

opt_dices = np.array(dice_records[best_t])
print('\nPerformance Breakdown by Slice Tumor Area:')
if small_mask.sum() > 0:
    print(f'  Small Tumors (<2k px @ 512, n={small_mask.sum()}):  Mean Dice = {opt_dices[small_mask].mean()*100:.2f}%')
if medium_mask.sum() > 0:
    print(f'  Medium Tumors (2k-10k px @ 512, n={medium_mask.sum()}): Mean Dice = {opt_dices[medium_mask].mean()*100:.2f}%')
if large_mask.sum() > 0:
    print(f'  Large Tumors (>10k px @ 512, n={large_mask.sum()}):  Mean Dice = {opt_dices[large_mask].mean()*100:.2f}%')

## Cell 10 — Visualize Test Predictions (512x512)

In [ ]:
import random
import matplotlib.pyplot as plt

test_dataset = test_loader.dataset
indices = random.sample(range(len(test_dataset)), 4)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Baseline U-Net (512x512) — High-Resolution Test Predictions', fontsize=16, fontweight='bold')

for row, idx in enumerate(indices):
    img_tensor, mask_tensor = test_dataset[idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_input)
        pred = torch.sigmoid(logits).squeeze().cpu().numpy()

    # Image for plotting
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    gt_mask = mask_tensor.squeeze().cpu().numpy()

    # MRI input
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'MRI Input 512x512 (Sample #{idx})')
    axes[row, 0].axis('off')

    # Ground truth
    axes[row, 1].imshow(gt_mask, cmap='gray')
    axes[row, 1].set_title('Ground Truth Mask')
    axes[row, 1].axis('off')

    # Predicted mask
    axes[row, 2].imshow(pred > best_t, cmap='hot')
    axes[row, 2].set_title(f'U-Net 512 Pred (t={best_t:.2f}, max={pred.max():.2f})')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()